In [1]:
# --- replication package paths (auto-inserted) ---
from pathlib import Path

# Resolve the package root whether run from notebooks/ or the root.
_here = Path.cwd()
ROOT = _here if (_here / "data").exists() else _here.parent

DATA    = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

assert DATA.exists(), f"data folder not found: {DATA}"


In [2]:
"""
H5 Analysis: License Friction Patterns Across Ownership Cohorts
------------------------------------------------------------------
H5. Method-level license compatibility friction patterns differ across
Corporate, OSS Foundation, and Individual repository ownership cohorts.

Design notes:
- project_type is attached per relationship row directly, so no
  URL-based project-identity resolution is needed here, unlike H1/H2.
- Friction here means categories 3 (incompatible), 4 (high-risk) AND
  5 (undetermined/latent). This is BROADER than H2/H4, which use 3 and
  4 only. H5 asks whether ownership predicts license-handling
  discipline, and unresolved licensing metadata is itself informative
  about governance quality rather than a neutral gap. Categories 1
  (same-license) and 2 (compatible) are NON-FRICTION.
- Category 0 (no provenance match) is excluded: no license comparison
  occurred, so friction status is undefined for these rows, not
  merely "non-friction."

Reproduces: N = 868,369; friction rates 66.22 / 75.51 / 77.35 percent
(Corporate / Individual / OSS Foundation); chi2(2) = 11,179.39,
Cramer's V = 0.114.
"""

import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations

# =============================================================================
# CONFIGURATION
# =============================================================================

RELATIONSHIPS_CSV = DATA / "license_analysis_results_processed.csv"
DEDUP_KEYS = ["method_hash", "source_repository_url", "sink_repository_url"]

PROJECT_TYPE_LABELS = {1: "Corporate", 2: "OSS Foundation", 3: "Individual"}

FRICTION_CATEGORIES = {3, 4, 5}
NON_FRICTION_CATEGORIES = {1, 2}
EXCLUDED_CATEGORIES = {0}

ALPHA = 0.05


# =============================================================================
# Stage 1: Load + dedup
# =============================================================================

def load_relationships(path):
    df = pd.read_csv(path)
    before = len(df)
    df = df.drop_duplicates(subset=DEDUP_KEYS)
    print(f"[dedup] {before} raw rows -> {len(df)} after dedup "
          f"({before - len(df)} duplicates removed, {(before - len(df)) / before:.3%})")
    return df


# =============================================================================
# Stage 2: Build friction indicator + cohort label
# =============================================================================

def build_h5_dataframe(df):
    df = df.copy()

    n0 = len(df)
    df = df[~df["violation_lcd_category"].isin(EXCLUDED_CATEGORIES)]
    print(f"[filter] {n0} -> {len(df)} rows after excluding Category 0 "
          f"(no provenance match; friction undefined) "
          f"({n0 - len(df)} removed, {(n0 - len(df)) / n0:.3%})")

    df["friction"] = df["violation_lcd_category"].apply(
        lambda c: True if c in FRICTION_CATEGORIES
        else False if c in NON_FRICTION_CATEGORIES
        else np.nan
    )

    unmapped = df["friction"].isna().sum()
    if unmapped > 0:
        print(f"[warn] {unmapped} rows have a violation_lcd_category value outside "
              f"{FRICTION_CATEGORIES | NON_FRICTION_CATEGORIES} and will be dropped: "
              f"{df.loc[df['friction'].isna(), 'violation_lcd_category'].value_counts().to_dict()}")
    df = df.dropna(subset=["friction"])
    df["friction"] = df["friction"].astype(bool)

    df["cohort"] = df["project_type"].map(PROJECT_TYPE_LABELS)
    unmapped_cohort = df["cohort"].isna().sum()
    if unmapped_cohort > 0:
        print(f"[warn] {unmapped_cohort} rows have an unmapped project_type value: "
              f"{df.loc[df['cohort'].isna(), 'project_type'].value_counts().to_dict()}")
    df = df.dropna(subset=["cohort"])

    return df


# =============================================================================
# Stage 3: Chi-square test of independence + Cramer's V
# =============================================================================

def cramers_v(contingency_table):
    """Cramer's V effect size for an r x c contingency table."""
    chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    r, c = contingency_table.shape
    phi2 = chi2 / n
    phi2_corr = max(0, phi2 - ((c - 1) * (r - 1)) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    c_corr = c - ((c - 1) ** 2) / (n - 1)
    v = np.sqrt(phi2_corr / min(c_corr - 1, r_corr - 1))
    return v, chi2, p, dof


def run_omnibus_test(df):
    contingency = pd.crosstab(df["cohort"], df["friction"])
    contingency = contingency.reindex(columns=[False, True], fill_value=0)
    contingency.columns = ["Non-Friction", "Friction"]

    print("\n=== Contingency Table (Cohort x Friction) ===")
    print(contingency)
    print("\n=== Friction Rate by Cohort ===")
    friction_rate = contingency["Friction"] / contingency.sum(axis=1)
    print((friction_rate * 100).round(2).astype(str) + "%")

    v, chi2, p, dof = cramers_v(contingency)
    print(f"\nChi-square({dof}) = {chi2:.2f}, p = {p:.4g}, Cramer's V = {v:.4f}")

    return {"contingency": contingency, "chi2": chi2, "p_value": p,
            "dof": dof, "cramers_v": v, "friction_rate": friction_rate}


# =============================================================================
# Stage 4: Post-hoc pairwise comparisons
# =============================================================================

def holm_bonferroni(pvalues):
    pvalues = np.asarray(pvalues)
    order = np.argsort(pvalues)
    adjusted = np.empty_like(pvalues)
    n = len(pvalues)
    running_max = 0
    for rank, idx in enumerate(order):
        adj = (n - rank) * pvalues[idx]
        running_max = max(running_max, adj)
        adjusted[idx] = min(running_max, 1.0)
    return adjusted


def run_pairwise_posthoc(df):
    cohorts = sorted(df["cohort"].unique())
    results = []

    for c1, c2 in combinations(cohorts, 2):
        subset = df[df["cohort"].isin([c1, c2])]
        contingency = pd.crosstab(subset["cohort"], subset["friction"])
        contingency = contingency.reindex(columns=[False, True], fill_value=0)
        v, chi2, p, dof = cramers_v(contingency)
        results.append({
            "comparison": f"{c1} vs {c2}",
            "chi2": chi2, "p_value": p, "cramers_v": v,
            "n1": len(subset[subset["cohort"] == c1]),
            "n2": len(subset[subset["cohort"] == c2]),
        })

    results_df = pd.DataFrame(results)
    results_df["p_adj"] = holm_bonferroni(results_df["p_value"].values)
    results_df["significant"] = results_df["p_adj"] < ALPHA

    print("\n=== Post-hoc Pairwise Comparisons (Holm-Bonferroni adjusted) ===")
    print(results_df.to_string(index=False))

    return results_df


# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":
    raw = load_relationships(RELATIONSHIPS_CSV)
    df = build_h5_dataframe(raw)

    print(f"\nN = {len(df)} relationships with resolvable friction status and cohort")
    print(df["cohort"].value_counts())

    omnibus = run_omnibus_test(df)
    posthoc = run_pairwise_posthoc(df)

    omnibus["contingency"].to_csv(RESULTS / "h5_contingency_table.csv")
    omnibus["friction_rate"].to_csv(RESULTS / "h5_friction_rate_by_cohort.csv")
    posthoc.to_csv(RESULTS / "h5_posthoc_pairwise.csv", index=False)

    print("\n[done] Results written to h5_contingency_table.csv, "
          "h5_friction_rate_by_cohort.csv, h5_posthoc_pairwise.csv")

[dedup] 1183182 raw rows -> 1183182 after dedup (0 duplicates removed, 0.000%)
[filter] 1183182 -> 868369 rows after excluding Category 0 (no provenance match; friction undefined) (314813 removed, 26.607%)

N = 868369 relationships with resolvable friction status and cohort
cohort
Corporate         388116
Individual        250303
OSS Foundation    229950
Name: count, dtype: int64

=== Contingency Table (Cohort x Friction) ===
                Non-Friction  Friction
cohort                                
Corporate             131117    256999
Individual             61308    188995
OSS Foundation         52086    177864

=== Friction Rate by Cohort ===
cohort
Corporate         66.22%
Individual        75.51%
OSS Foundation    77.35%
dtype: object

Chi-square(2) = 11179.39, p = 0, Cramer's V = 0.1135

=== Post-hoc Pairwise Comparisons (Holm-Bonferroni adjusted) ===
                  comparison        chi2      p_value  cramers_v     n1     n2        p_adj  significant
     Corporate vs Ind

In [3]:
"""
H5 Clustering Robustness: cluster-robust inference + organization-level
aggregation
--------------------------------------------------------------------------
WHY NOT MIXED-EFFECTS

A Bayesian mixed GLM with organization random intercepts was attempted and
failed to converge: posterior SDs collapsed to 0.0000, the intercept implied
a 99.8% friction rate against ~71% observed, the random-effect SD reached
143 on the logit scale (CI [53, 383]), and the optimizer reported overflow.
Those estimates are unusable.

This script instead uses two stable approaches that address the same
reviewer concern -- that relationship-level observations cluster within
repositories and organizations, so naive p-values understate uncertainty:

  1. CLUSTER-ROBUST LOGIT. Keeps every observation but computes
     cluster-robust (sandwich) standard errors with organizations as
     clusters. Directly corrects the calibration of the standard errors
     without requiring a random-effects model to converge.

  2. ORGANIZATION-LEVEL AGGREGATION. Collapses to one friction rate per
     organization, making the ORGANIZATION the unit of analysis. This
     sidesteps within-organization clustering entirely: with ~52
     independent units, no observation shares a cluster with another.
     Non-parametric comparison (Mann-Whitney U + Cliff's delta), matching
     the effect-size-first methodology used elsewhere in the paper.

If the Corporate-vs-OSS-Foundation contrast survives both, H5's conclusion
is robust to clustering. If it weakens sharply under aggregation, the
cohort effect is driven by a few large organizations and must be qualified.
"""

import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import mannwhitneyu

FILE_PATH = DATA / "license_analysis_results_processed.csv"
MIN_RELATIONSHIPS_PER_ORG = 30
PSEUDO_ORGS = {"Individual", "individual", "Unknown", "N/A"}
FRICTION_CATEGORIES = [3, 4, 5]      # H5's definition (see paper, Study Design)
COHORT_MAP = {1: "Corporate", 2: "OSS Foundation", 3: "Individual"}


def cliffs_delta(a, b):
    """Cliff's delta: P(a>b) - P(a<b), computed exactly."""
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((a[:, None] > b[None, :]).sum(axis=1))
    lt = sum((a[:, None] < b[None, :]).sum(axis=1))
    return (gt - lt) / (len(a) * len(b))


def build(path=FILE_PATH):
    df = pd.read_csv(path)
    df = df.drop_duplicates(subset=["method_hash", "source_repository_url",
                                    "sink_repository_url"])
    df = df[df["violation_lcd_category"] != 0]
    df["is_friction"] = df["violation_lcd_category"].isin(FRICTION_CATEGORIES).astype(int)
    df["cohort"] = df["project_type"].map(COHORT_MAP)
    df = df.dropna(subset=["cohort", "organization_name"])

    rates = df.groupby("cohort")["is_friction"].mean().mul(100).round(2)
    print(f"N = {len(df):,}  |  friction by cohort: {rates.to_dict()}")
    print("Expected: {'Corporate': 66.22, 'Individual': 75.51, 'OSS Foundation': 77.35}")
    ok = abs(rates.get("Corporate", 0) - 66.22) < 0.05
    print("[ok] reproduces published H5\n" if ok else "[!] MISMATCH -- stop\n")
    return df


def run(path=FILE_PATH):
    df = build(path)

    # Restrict to organizations with genuine institutional identity.
    df = df[~df["organization_name"].isin(PSEUDO_ORGS)].copy()
    counts = df["organization_name"].value_counts()
    df = df[df["organization_name"].isin(counts[counts >= MIN_RELATIONSHIPS_PER_ORG].index)]
    df["is_corporate"] = (df["cohort"] == "Corporate").astype(int)
    n_org = df["organization_name"].nunique()
    print(f"Restricted to real organizations: {len(df):,} relationships, {n_org} organizations")
    print(f"(fixed effect contrasts Corporate vs OSS Foundation)\n")

    y = df["is_friction"]
    X = sm.add_constant(df[["is_corporate"]])

    # ---- 1. Naive vs cluster-robust logit -------------------------------
    print("=" * 68)
    print("1. LOGIT: naive vs cluster-robust standard errors")
    print("=" * 68)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        naive = sm.Logit(y, X).fit(disp=False)
        clust = sm.Logit(y, X).fit(
            cov_type="cluster",
            cov_kwds={"groups": df["organization_name"].astype("category").cat.codes},
            disp=False)

    rows = []
    for name, m in [("Naive (independence assumed)", naive),
                    ("Cluster-robust (by organization)", clust)]:
        b, se, p = m.params["is_corporate"], m.bse["is_corporate"], m.pvalues["is_corporate"]
        rows.append({"model": name, "coef": round(b, 4), "SE": round(se, 4),
                     "OR": round(np.exp(b), 4), "p": f"{p:.4g}",
                     "sig@.05": p < .05})
    print(pd.DataFrame(rows).to_string(index=False))
    infl = clust.bse["is_corporate"] / naive.bse["is_corporate"]
    print(f"\nSE inflation from clustering: {infl:.2f}x")
    print("(>1 confirms naive SEs were too small, as the reviewer anticipated.)")

    # ---- 2. Organization-level aggregation ------------------------------
    print("\n" + "=" * 68)
    print("2. ORGANIZATION-LEVEL AGGREGATION (organization = unit of analysis)")
    print("=" * 68)
    org = (df.groupby(["organization_name", "cohort"])
             .agg(friction_rate=("is_friction", "mean"),
                  n_rel=("is_friction", "size"))
             .reset_index())
    corp = org[org["cohort"] == "Corporate"]["friction_rate"].values
    ossf = org[org["cohort"] == "OSS Foundation"]["friction_rate"].values

    print(f"Corporate organizations     : n = {len(corp):>3}, "
          f"median friction {np.median(corp):.3f}, mean {corp.mean():.3f}")
    print(f"OSS Foundation organizations: n = {len(ossf):>3}, "
          f"median friction {np.median(ossf):.3f}, mean {ossf.mean():.3f}")

    if len(corp) >= 3 and len(ossf) >= 3:
        u, p = mannwhitneyu(corp, ossf, alternative="two-sided")
        d = cliffs_delta(corp, ossf)
        mag = ("negligible" if abs(d) < .147 else "small" if abs(d) < .33
               else "medium" if abs(d) < .474 else "large")
        print(f"\nMann-Whitney U = {u:.1f}, p = {p:.4g}")
        print(f"Cliff's delta  = {d:.3f} ({mag})")
        print("\n=> " + ("Corporate advantage holds at the organization level."
                         if p < .05 else
                         "NOT significant at the organization level -- the "
                         "cohort effect may be driven by a few large organizations."))
    else:
        print("\n[!] Too few organizations per cohort for a meaningful test.")

    org.sort_values("friction_rate").to_csv(RESULTS / "h5_org_level_rates.csv", index=False)
    print("\nSaved per-organization rates: h5_org_level_rates.csv")
    return org


if __name__ == "__main__":
    run()

N = 868,369  |  friction by cohort: {'Corporate': 66.22, 'Individual': 75.51, 'OSS Foundation': 77.35}
Expected: {'Corporate': 66.22, 'Individual': 75.51, 'OSS Foundation': 77.35}
[ok] reproduces published H5

Restricted to real organizations: 618,001 relationships, 52 organizations
(fixed effect contrasts Corporate vs OSS Foundation)

1. LOGIT: naive vs cluster-robust standard errors
                           model    coef    SE    OR      p  sig@.05
    Naive (independence assumed) -0.5551 0.006 0.574      0     True
Cluster-robust (by organization) -0.5551 0.627 0.574 0.3759    False

SE inflation from clustering: 104.00x
(>1 confirms naive SEs were too small, as the reviewer anticipated.)

2. ORGANIZATION-LEVEL AGGREGATION (organization = unit of analysis)
Corporate organizations     : n =  32, median friction 0.726, mean 0.660
OSS Foundation organizations: n =  20, median friction 0.857, mean 0.764

Mann-Whitney U = 247.0, p = 0.1727
Cliff's delta  = -0.228 (small)

=> NOT signif

In [4]:
"""
VERIFICATION CELL -- independent cross-check of the contingency table
================================================================================
Recomputes cohort x friction directly via pd.crosstab, bypassing the main
pipeline, and asserts the result against the published rates.

Expected: 66.22 / 75.51 / 77.35, overall 71.84%.
A mismatch here means the friction definition or the Category 0 filter has
drifted from the specification in the paper's Study Design section.
================================================================================
"""

import pandas as pd

FILE_PATH = DATA / "license_analysis_results_processed.csv"

raw = pd.read_csv(FILE_PATH)
df = raw.drop_duplicates(subset=["method_hash", "source_repository_url", "sink_repository_url"])
df = df[df["violation_lcd_category"] != 0]

print("=== dtype and raw value check ===")
print("dtype:", df["violation_lcd_category"].dtype)
print("unique values:", sorted(df["violation_lcd_category"].unique().tolist()))
print()

df["is_friction"] = df["violation_lcd_category"].isin([3, 4, 5]).astype(int)
print("=== is_friction value_counts (should NOT be near-all-zero) ===")
print(df["is_friction"].value_counts())
print(f"Overall friction rate: {df['is_friction'].mean():.2%} (expected ~71.85%)")
print()

cohort_map = {1: "Corporate", 2: "OSS Foundation", 3: "Individual"}
df["cohort"] = df["project_type"].map(cohort_map)
df = df.dropna(subset=["cohort", "organization_name"])

print("=== Direct crosstab: cohort x is_friction (no model, ground truth) ===")
ct = pd.crosstab(df["cohort"], df["is_friction"])
print(ct)
print()
print("=== Friction rate by cohort, computed directly ===")
print((ct[1] / (ct[0] + ct[1])).round(4))
print("Expected: Corporate=0.6622  Individual=0.7551  OSS Foundation=0.7735")

=== dtype and raw value check ===
dtype: int64
unique values: [1, 2, 3, 4, 5]

=== is_friction value_counts (should NOT be near-all-zero) ===
is_friction
1    623858
0    244511
Name: count, dtype: int64
Overall friction rate: 71.84% (expected ~71.85%)

=== Direct crosstab: cohort x is_friction (no model, ground truth) ===
is_friction          0       1
cohort                        
Corporate       131117  256999
Individual       61308  188995
OSS Foundation   52086  177864

=== Friction rate by cohort, computed directly ===
cohort
Corporate         0.6622
Individual        0.7551
OSS Foundation    0.7735
dtype: float64
Expected: Corporate=0.6622  Individual=0.7551  OSS Foundation=0.7735
